# QAOAを用いてMax Cust問題を解く

ここでは、量子コンピュータを用いて、Max Cut問題をQAOA（Quantum Approximate Optimization Algorithm：量子近似最適化アルゴリズム）を用いて解きます。

本サンプルはABCI-Q上で動作します。具体的にはOpen Ondemandから起動するCovalentのJupyter Notebook環境で動作します。詳細な利用方法は、ABCI-Q User Guideの[Covalentマニュアル](https://g-quat-abciq.github.io/abciq-docs/ja/apps/covalent/)をご覧ください。

本サンプルは「[QAOA徹底解説：Max-Cutをステップ・バイ・ステップで解く](https://www.freequantumcomputing.com/ja/tutorials/qaoa-max-cut-tutorial/)」のチュートリアルを参考に、quri-parts-oqtopusを用いるように変更しCovalentに対応したものです。


### 1. 必要なライブラリのimport

In [1]:
import covalent as ct

import numpy as np
from scipy.optimize import minimize
from quri_parts.circuit import QuantumCircuit, Parameter
from quri_parts_oqtopus.backend import OqtopusConfig, OqtopusSamplingBackend

### 2. 量子回路の作成

シンプルな４ノードのグラフを定義し、`create_qaoa_circuit`で２つのパラメータを与えるQAOAの回路を構築します。

この回路は`sample`関数にて、quri-parts-oqtopusを使用し、ABCI-Q上で量子コンピュータ（またはシミュレータ）で実行されます。

QAOA回路に渡すパラメータは`scipy`ライブラリの`minimize`関数を使って最適化されます。`objective`関数では、回路実行によって得られたサンプル結果からMax-Cutの期待値を計算し、その値が最大となるようにパラメータ探索を行います。

In [2]:
# =====================================
#  Max Cut Edges
# =====================================
EDGES = [
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 0),
]

N_QUBITS = 4

# =====================================
#  Max Cut QAOA Circuit
# =====================================
def create_qaoa_circuit(gamma: float, beta: float) -> QuantumCircuit:
    circuit = QuantumCircuit(N_QUBITS)

    for q in range(N_QUBITS):
        circuit.add_H_gate(q)

    # Cost Unitary
    for i, j in EDGES:
        circuit.add_CNOT_gate(i, j)
        circuit.add_RZ_gate(j, gamma)
        circuit.add_CNOT_gate(i, j)

    # Mixer Unitary
    for q in range(N_QUBITS):
        circuit.add_RX_gate(q, 2.0 * beta)

    return circuit


def bitstring_from_int(value: int) -> str:
    return format(value, f"0{N_QUBITS}b")


def maxcut_score(bitstring: str) -> int:
    score = 0

    for i, j in EDGES:
        if bitstring[i] != bitstring[j]:
            score += 1

    return score

def compute_expected_cost(counts: dict[int, int]) -> float:
    total_shots = sum(counts.values())

    if total_shots == 0:
        raise ValueError("No samples were returned.")

    expectation = 0.0

    for state, count in counts.items():
        bitstring = bitstring_from_int(state)
        cost = maxcut_score(bitstring)
        expectation += cost * count

    return -(expectation / total_shots)

def sample(circuit: QuantumCircuit, shots: int) -> dict[int, int]:
    backend = OqtopusSamplingBackend(OqtopusConfig.from_file("oqtopus-dev"))

    job = backend.sample(
        circuit,
        device_id="qiskit_sim",
        shots=shots,
    )

    return dict(job.result().counts)   

def objective(params):
    gamma, beta = params

    circuit = create_qaoa_circuit(gamma, beta)
    
    counts = sample(circuit, 2000)

    expectation = compute_expected_cost(counts)

    return expectation

@ct.electron
def optimize_electron():
    constraints = [
        {"type": "ineq", "fun": lambda x: x[0]},              # gamma >= 0
        {"type": "ineq", "fun": lambda x: np.pi - x[0]},      # gamma <= pi
        {"type": "ineq", "fun": lambda x: x[1]},              # beta >= 0
        {"type": "ineq", "fun": lambda x: np.pi / 2 - x[1]},  # beta <= pi/2
    ]

    initial_params = [0.7, 0.4]

    result = minimize(
        objective,
        initial_params,
        method="COBYLA",
        constraints=constraints,
        options={
            "maxiter": 50,
            "disp": True,
        },
    )

    return result

### 3. 最良解による最終サンプリング

`final_sampling`では引数で渡された最適解パラメータを使って、量子コンピュータで最終サンプリングを実行して結果を得ます。
ここではCovalentのSystem-Fプラグインを使用し単独のタスクとして実行します。

In [3]:
qps = ct.executor.AbciqSysFSamplingExecutor(
    device_id = "qiskit_sim",     # Enter the device ID of Quantum PC.
    shots = 10000,          # Number of shots.
    timeout = 3600,               # Timeout to get procedure result (second).
    interval = 60,                # Time interval to query to get status (second).
    use_oqtopus_config = True,    # Flag of use to oqtopus config file.
    oqtopus_section = "oqtopus-dev",     # when use oqtopus config file, select to section name.
    oqtopus_path = "~/.oqtopus",  # when use oqtopus config file, enter oqtopus config file path name including filename.
)

@ct.electron(executor=qps)
def final_sampling(result):
    gamma_opt, beta_opt = result['x']

    circuit = create_qaoa_circuit(
        gamma_opt,
        beta_opt,
    )

    param = {
        "circuit": circuit.freeze(),
    }

    return param

### 4. Covalentのワークフローの定義と実行

ワークフローを定義して実行します。
Covalentのタスクとしては、量子回路のパラメータ最適化を行う`optimize_electoin`と`final_sampling`の２つになります。

In [4]:
@ct.lattice
def workflow():
    result = optimize_electron()
    counts = final_sampling(result)
    
    return counts

dispatch_id = ct.dispatch(workflow)()

### 5. 最終サンプリングの結果の出力

サンプリングの結果をソートして、カウント数の多い順に出力させことで、最も可能性の高い良い解を見つけます。

In [5]:
def print_top_samples(counts: dict[int, int], top_k: int = 10) -> None:
    sorted_counts = sorted(
        counts.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    for state, count in sorted_counts[:top_k]:
        bitstring = bitstring_from_int(state)
        score = maxcut_score(bitstring)

        print(
            f"{bitstring} : "
            f"count={count} "
            f"cut={score}"
        )

workflow_job = ct.get_result(dispatch_id, wait=True)
print_top_samples(workflow_job.result.counts)

0101 : count=2644 cut=4
1010 : count=2577 cut=4
0011 : count=968 cut=2
1100 : count=894 cut=2
0110 : count=855 cut=2
1001 : count=828 cut=2
0000 : count=194 cut=0
1111 : count=171 cut=0
0001 : count=120 cut=2
0111 : count=118 cut=2
